# 反向传播

## 学习目标

串联 Linear、ReLU 与交叉熵，完成一次完整的前向和反向传播。

## 概念模型

每层只需要知道自己的输入和上游梯度。反向传播按前向的相反顺序调用，各层把梯度交给前一层。

## 逐步实现

按顺序运行下面的代码，并在每一步检查 shape、数值范围和中间结果。

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "07-deep-learning/fundamentals"
sys.path.insert(0, str(course_dir.resolve()))

import numpy as np
from from_scratch import CrossEntropyLoss, Linear, ReLU

x = np.array([[0.2, -0.4], [1.0, 0.3], [-0.5, 0.8]])
y = np.array([0, 1, 0])
layer1, activation, layer2 = Linear(2, 4, 1), ReLU(), Linear(4, 2, 2)

hidden = layer1.forward(x)
activated = activation.forward(hidden)
logits = layer2.forward(activated)
loss_fn = CrossEntropyLoss()
loss = loss_fn.forward(logits, y)
print("loss:", loss)

In [ ]:
gradient = loss_fn.backward()
gradient = layer2.backward(gradient)
gradient = activation.backward(gradient)
gradient = layer1.backward(gradient)
print("dX shape:", gradient.shape)
print("first layer gradient norm:", np.linalg.norm(layer1.grad_weight))
assert gradient.shape == x.shape

## 检查点

梯度流向为 loss → logits → hidden → inputs；每个可训练层同时保存参数梯度，供优化器更新。

## 试一试

交换两个 backward 调用，预测会出现哪一种 shape 错误，再恢复正确顺序。

## 常见错误

反向顺序与前向相同；用更新后的权重继续传播旧梯度；批次损失求均值后又重复除以 batch size。